In [32]:
import argparse
import copy
import json
import os.path
from enum import Enum
from typing import Dict, Tuple, Union, Optional, Any
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from PIL import Image
from typing import List, Iterable, Optional, Union
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import os
# physical_devices = tf.config.list_physical_devices('GPU')
# tf.config.experimental.set_memory_growth(physical_devices[0], True)
from numpy import ndarray
from tensorflow import keras, Tensor
from tensorflow.keras.layers import Conv2D
from tensorflow.python.keras.models import Functional
from keras_models import generate_ncp_model
# from vis_utils import run_visualization, write_video

IMAGE_SHAPE = (144, 256, 3)
IMAGE_SHAPE_CV = (IMAGE_SHAPE[1], IMAGE_SHAPE[0])

DEFAULT_NCP_SEED = 22222

batch_size = None
seq_len = 64
augmentation_params = None
single_step = True
no_norm_layer = False
mymodel = generate_ncp_model(seq_len, IMAGE_SHAPE, augmentation_params, batch_size, DEFAULT_NCP_SEED, single_step, no_norm_layer)


# custom model weights

mymodel.load_weights('../saved_models/retrain_mix_goal_heights_diff_coreset_wscheduler0.85_seed22222_lr0.001_trainloss0.00008_epoch100.h5')

In [33]:
mymodel.summary()

Model: "model_14"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_16 (InputLayer)           [(None, 144, 256, 3) 0                                            
__________________________________________________________________________________________________
rescaling_8 (Rescaling)         (None, 144, 256, 3)  0           input_16[0][0]                   
__________________________________________________________________________________________________
normalization_8 (Normalization) (None, 144, 256, 3)  7           rescaling_8[0][0]                
__________________________________________________________________________________________________
conv2d_40 (Conv2D)              (None, 70, 126, 24)  1824        normalization_8[0][0]            
___________________________________________________________________________________________

In [34]:
for idx, layer in enumerate(mymodel.layers):
    print(f"{idx}: {layer.name} ({layer.__class__.__name__}) -> Output shape: {layer.output_shape}")

0: input_16 (InputLayer) -> Output shape: [(None, 144, 256, 3)]
1: rescaling_8 (Rescaling) -> Output shape: (None, 144, 256, 3)
2: normalization_8 (Normalization) -> Output shape: (None, 144, 256, 3)
3: conv2d_40 (Conv2D) -> Output shape: (None, 70, 126, 24)
4: conv2d_41 (Conv2D) -> Output shape: (None, 33, 61, 36)
5: conv2d_42 (Conv2D) -> Output shape: (None, 15, 29, 48)
6: conv2d_43 (Conv2D) -> Output shape: (None, 13, 27, 64)
7: conv2d_44 (Conv2D) -> Output shape: (None, 6, 13, 16)
8: flatten_8 (Flatten) -> Output shape: (None, 1248)
9: dense_8 (Dense) -> Output shape: (None, 128)
10: dropout_8 (Dropout) -> Output shape: (None, 128)
11: input_17 (InputLayer) -> Output shape: [(None, 34)]
12: ltc_cell (LTCCell) -> Output shape: ((None, 4), [(None, 34)])


In [31]:
from tensorflow.keras.models import Model

# Get first input (image input)
first_input = mymodel.inputs[0]

# Get output up to flatten layer (index 8)
last_layer_output = mymodel.layers[8].output

# Build the new model
model_feature_extraction_till_flatten = Model(inputs=first_input, outputs=last_layer_output)

# Verify
model_feature_extraction_till_flatten.summary()

Model: "model_13"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_14 (InputLayer)        [(None, 144, 256, 3)]     0         
_________________________________________________________________
rescaling_7 (Rescaling)      (None, 144, 256, 3)       0         
_________________________________________________________________
normalization_7 (Normalizati (None, 144, 256, 3)       7         
_________________________________________________________________
conv2d_35 (Conv2D)           (None, 70, 126, 24)       1824      
_________________________________________________________________
conv2d_36 (Conv2D)           (None, 33, 61, 36)        21636     
_________________________________________________________________
conv2d_37 (Conv2D)           (None, 15, 29, 48)        43248     
_________________________________________________________________
conv2d_38 (Conv2D)           (None, 13, 27, 64)        277

In [35]:
def load_image(image_path):
    img = Image.open(image_path)
    img = img.resize(IMAGE_SHAPE_CV)
    img_array = np.array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = tf.convert_to_tensor(img_array)
    return img_array

img = load_image('../../fly_to_target_dataset/diff_dataset/1/Image500.png')

ideal_feature_vector = model_feature_extraction_till_flatten.predict(img)

current_img = load_image('../../fly_to_target_dataset/diff_dataset/1/Image1.png')
current_feature_vector = model_feature_extraction_till_flatten.predict(current_img)

feature_diff = ideal_feature_vector - current_feature_vector

In [41]:
val_set = set(feature_diff[0])
print(len(val_set))

236


In [42]:
from tensorflow.keras.models import Model
import numpy as np

# Step 1: Get Dense and Dropout output after flatten
dense_input = mymodel.layers[8].output  # flatten_*
dense_output = mymodel.layers[9](dense_input)  # dense_*
dropout_output = mymodel.layers[10](dense_output)  # dropout_*

# Step 2: Get LTC layer
ltc_layer = mymodel.get_layer(index=12)  # ltc_cell

# ⚠ LTC expects two inputs: from dropout and from input_17
# To bypass input_17, you **must** provide something (e.g., zeros)
# Build the partial model
dense_input_tensor = mymodel.inputs[0]  # first input (image) is NOT used, we'll pass feature_diff manually
second_input_tensor = mymodel.inputs[1]  # second input, needed by ltc_cell

# Build model from dense/dropout + second input
partial_model = Model(inputs=[dense_output, second_input_tensor], outputs=ltc_layer.output)

# Step 3: Prepare dummy input for input_17
batch_size = feature_diff.shape[0]
dummy_input_17 = np.zeros((batch_size, 34))  # shape must match input_17

# Step 4: Pass feature_diff through remaining layers
final_output = partial_model.predict([feature_diff, dummy_input_17])

print("Final Output:", final_output)


Note that input tensors are instantiated via `tensor = tf.keras.Input(shape)`.
The tensor that caused the issue was: dense_8/BiasAdd:0


AssertionError: 